# ADT-intent-stratified survival models

Univariate Cox, elastic-net Cox, and XGBoost survival models for progression to platinum, NEPC, and AVPC, fit separately within the medication-derived `METASTATIC` and `LOCALIZED_ADJUVANT` strata.

Run `01_preprocessing.ipynb` first. This notebook reuses the ordinary ADT-anchored Stage-2 frame and creates new Stage-3 input/output trees; it does not rebuild the cohort or labs.

**Interpretation warning:** `ADT_INTENT` uses the full observed ADT course (duration, cessation/restart, and later definitive escalation). Platinum is excluded from the classifier, but the stratum is still not available prospectively at the landmark. Results are retrospective within-stratum associations, not validated baseline predictions of metastatic status.

In [ ]:
ENDPOINTS = ("platinum", "nepc", "avpc")
STRATA = ("metastatic", "localized")
REBUILD_PREDICTION_INPUTS = True
OVERWRITE_MODELS = False

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

cp.N_FOLDS = 5
cp.FORCE_RERUN = OVERWRITE_MODELS

## Build the medication-derived model strata

Writes a combined audit label, one MRN list per stratum, and preliminary incident endpoint counts under `<data_root>/mrn_lists/`. The count table is an early warning for localized endpoints with too few events to fit reliably.

In [ ]:
STRATUM_FILES = cp.build_adt_intent_mrn_lists()
RUNS = cp.make_adt_intent_endpoint_runs(strata=STRATA, endpoints=ENDPOINTS)
STRATUM_FILES

In [ ]:
import pandas as pd

preliminary_counts = pd.read_csv(STRATUM_FILES["counts"])
preliminary_counts

## Build endpoint-specific prediction inputs

Each stratum and endpoint gets an independent cohort, split, canonical lab set, and output tree. Endpoint eligibility remains independent (`t_platinum`, `t_nepc`, or `t_avpc` only), as in the main pipeline.

In [ ]:
for run in RUNS:
    if REBUILD_PREDICTION_INPUTS:
        cp.build_prediction_inputs(run)
    else:
        print(f"[skip] input rebuild: {run['label']} / {run['endpoint']}")
    cp.cohort_diagnostics(run)

## Univariate Cox associations

In [ ]:
for run in RUNS:
    cp.run_univariate(run)

In [ ]:
NOMINAL_ALPHA = 0.05
nominal_tables = {}

for run in RUNS:
    results = cp.load_univariate_results(run)
    filtered = cp.filter_nominal(results, alpha=NOMINAL_ALPHA)
    key = (run["adt_intent"], run["endpoint"])
    nominal_tables[key] = filtered
    export_path = run["output_dir"] / "cox" / "nominally_significant_univariate_results.csv"
    if not export_path.exists() or OVERWRITE_MODELS:
        export_path.parent.mkdir(parents=True, exist_ok=True)
        filtered.to_csv(export_path, index=False)
    print(f"{key}: {len(filtered)} nominal hits -> {export_path}")

nominal_tables

## Multivariate models

Runs elastic-net Cox and XGBoost survival:cox with both full and androgen-axis baseline feature sets at landmarks 0, 90, and 180 days.

In [ ]:
for run in RUNS:
    cp.run_multivariate(run)

In [ ]:
summary_dfs = {
    (run["adt_intent"], run["endpoint"]): cp.summarize_outputs(run)
    for run in RUNS
}
combined_summary_df = pd.concat(summary_dfs.values(), ignore_index=True)
combined_summary_df